# US Quarterly Earnings Data 2019–2026: 7.6k Stocks: starter notebook

This file is a **sample**. The full data, the withheld fields and monthly updates come from the [Radar API](https://github.com/fliarbi/radar-api-mcp). This notebook loads the sample, explores it, then gets one record's full detail from the API.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
path = (glob.glob('/kaggle/input/**/us_quarterly_results_sample.csv', recursive=True) or ['us_quarterly_results_sample.csv'])[0]
df = pd.read_csv(path, low_memory=False)
print(df.shape); df.head()

In [ ]:
df.describe(include="all").T.head(30)

In [ ]:
df['period_end'] = pd.to_datetime(df.period_end)
q = df.groupby('calendar_quarter').agg(companies=('cik', 'nunique'), revenue_bn=('revenue_usd', lambda x: x.sum() / 1e9),
                                        profitable=('net_income_usd', lambda x: (x > 0).mean()))
q.tail(12)

In [ ]:
g = df[df.revenue_yoy_pct.notna() & (df.revenue_usd > 1e8)]
g.groupby('calendar_quarter').revenue_yoy_pct.median().plot(figsize=(10, 4), title='Median revenue growth YoY, companies with $100M+ quarterly revenue (sample)')

In [ ]:
df[df.calendar_quarter == df.calendar_quarter.max()].nlargest(20, 'revenue_usd')[['company_name', 'ticker', 'sector', 'fiscal_quarter', 'revenue_usd', 'net_income_usd']]

## Get the full record from the Radar API

The sample keeps three headline lines per quarter. The API returns the full income statement, balance sheet and cash flow for each quarter and year. [Create a key](https://fliarbi.com/radar-api/developer), then add it to Kaggle **Add-ons → Secrets** as `RADAR_API_KEY`.

In [ ]:
import os, json, requests
key = os.environ.get('RADAR_API_KEY')
try:
    from kaggle_secrets import UserSecretsClient
    key = key or UserSecretsClient().get_secret('RADAR_API_KEY')
except Exception:
    pass
value = df['company_domain'].dropna().astype(str).iloc[0]
if key:
    r = requests.post('https://radar-api.fliarbi.com/v1/companies/financials', headers={'Authorization': f'Bearer {key}'},
                      json={'period_type': 'quarterly', 'domain': value}, timeout=60)
    print(json.dumps(r.json(), indent=1)[:3000])
else:
    print('Set RADAR_API_KEY to fetch the full record for', value)

Each row also has `fliar_profile_url`, the company's full profile. Repository, docs and MCP setup: https://github.com/fliarbi/radar-api-mcp